In [7]:
"""
This script reads geo data from a CSV file and processes it to create a rectangular grid of values.
The grid is then plotted as an image overlay on a map using matplotlib.

Steps:
1. Read the data from the CSV file.
2. Calculate the highest class per point.
3. Get the x and y values.
4. Get the unique x and y values.
5. Convert the coordinates to a different coordinate system.
6. Calculate the bounds of the data.
7. Create a 3D grid to store the class values.
8. Fill the 3D grid with the class values.
9. Create a 2D grid to store the index of the class with the highest value.
10. Create a 2D grid to store the maximum value.
11. Create a list of colormaps.
12. Create an empty image.
13. Color the points in the image based on the class values.
14. Plot the image.
15. Create a legend.
16. Save the plot as an image file.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches
from transformation import transform_csv

# read the data
df = pd.read_csv('upload_data/8-col-31468.csv')

# get then highest class per point
df['highest_class'] = df[['Class1', 'Class2', 'Class3', 'Class4', 'Class5', 'Class6']].idxmax(axis=1)

# get the x and y values
x = df['Easting (m)'].values
y = df['Northing (m)'].values

# get the unique values
x_unique = np.unique(x)
y_unique = np.unique(y)

# get the number of unique values
x_unique_len = len(x_unique)
y_unique_len = len(y_unique)

# convert the coordinates to a different coordinate system to get the bounds for the dash leaflet map
trans_df = transform_csv('upload_data/8-col-31468.csv', 31468, 4326)
# get the bounds
x_min = trans_df['Latitude'].min()
x_max = trans_df['Latitude'].max()
y_min = trans_df['Longitude'].min()
y_max = trans_df['Longitude'].max()
print(f"old bounds: {x_min}, {x_max}, {y_min}, {y_max}")

# Create the 3D grid
classes = ['Class1', 'Class2', 'Class3', 'Class4', 'Class5', 'Class6']
grid_3d = np.zeros((x_unique_len, y_unique_len, len(classes)))

# Fill the 3D grid with the class values
for i in range(len(x)):
    x_index = np.where(x_unique == x[i])[0][0]
    y_index = np.where(y_unique == y[i])[0][0]
    for j, class_ in enumerate(classes):
        grid_3d[x_index, y_index, j] = df[class_][i]
        grid_3d[grid_3d == 0] = np.nan

# Create the 2D grid that stores the index of the class with the highest value
grid_max_class = np.argmax(grid_3d, axis=2)

# Create the 2D grid that stores the maximum value
grid_max_value = np.max(grid_3d, axis=2)

# Create the list of colormaps
cmaps = [plt.cm.Blues, plt.cm.Oranges, plt.cm.Greens, plt.cm.Purples, plt.cm.Reds, plt.cm.Greys]

# Create the empty image
image = np.zeros((x_unique_len, y_unique_len, 4))

# Color the points
for i, cmap in enumerate(cmaps):
    mask = (grid_max_class == i)
    image[mask] = cmap(grid_max_value[mask])

# Plot the image
fig, ax = plt.subplots()
ax.imshow(image, origin='lower')
ax.axis('off')

# Create the legend
patches = [mpatches.Patch(color=cmap(0.5), label=class_) for cmap, class_ in zip(cmaps, classes)]
ax.legend(handles=patches, loc='upper left', framealpha=0.5)

plt.savefig('assets/test_1.png', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

old bounds: 51.585032547449714, 51.889284921667524, 10.899554323779789, 11.518828435546714
